# LeetCode 2098 — Subsequence of Size K With the Largest Even Sum bằng Python

Link đề: https://leetcode.com/problems/subsequence-of-size-k-with-the-largest-even-sum/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

> **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def largestEvenSum(self, nums: List[int], k: int) -> int:
        ...
```

Lộ trình:
1. **Brute force** thử mọi tổ hợp `k` phần tử — `O(C(n, k) · k)`.
2. **DP theo tính chẵn lẻ**: `best[j][p]` = tổng lớn nhất khi chọn `j` phần tử với tổng có tính chẵn lẻ `p` — `O(n · k)`.
3. **Tham lam + một lần hoán đổi**: lấy `k` số lớn nhất; nếu tổng lẻ, sửa bằng **một** trong hai phép đổi — `O(n log n)`.
4. **Tách chẵn/lẻ + prefix sum**: thử mọi số lượng số lẻ **chẵn** `i`, lấy `i` số lẻ lớn nhất và `k − i` số chẵn lớn nhất.

## 1. Mục tiêu học

1. Nhận ra thứ tự trong "dãy con" **không quan trọng** — chỉ cần chọn một **tập** `k` phần tử.
2. Dùng tính chất chẵn lẻ: tổng chẵn ⇔ số phần tử **lẻ** được chọn là số **chẵn**.
3. Viết DP với trạng thái nhỏ `(số lượng, tính chẵn lẻ)` làm lời giải kiểm chứng.
4. Thiết kế tham lam "lấy lớn nhất rồi sửa" và chứng minh **một lần đổi là đủ**, phải xét **cả hai** kiểu đổi.
5. Biết lời giải tổng quát bằng prefix của hai nhóm, và biến thể "chia hết cho `m`".

## 2. Đề bài & ví dụ

Cho mảng `nums` và số nguyên `k`. Tìm **tổng chẵn lớn nhất** của một dãy con độ dài đúng `k`. Nếu không có dãy con độ dài `k` nào có tổng chẵn, trả về `-1`.

**Ví dụ 1:** `nums = [4,1,5,3,1]`, `k = 3` → `12` (`[4,5,3]`).

**Ví dụ 2:** `nums = [4,6,2]`, `k = 3` → `12`.

**Ví dụ 3:** `nums = [1,3,5]`, `k = 1` → `-1` (mọi phần tử đều lẻ).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `0 ≤ nums[i] ≤ 10⁵`, `1 ≤ k ≤ n`.

In [1]:
import random
import time
from itertools import combinations, accumulate
from typing import List

examples = [(([4, 1, 5, 3, 1], 3), 12), (([4, 6, 2], 3), 12), (([1, 3, 5], 1), -1)]
for (nums, k), want in examples:
    print(f"nums = {nums}, k = {k} -> {want}")

nums = [4, 1, 5, 3, 1], k = 3 -> 12
nums = [4, 6, 2], k = 3 -> 12
nums = [1, 3, 5], k = 1 -> -1


## 3. Kiến thức nền

### 3.1 Tính chẵn lẻ của tổng

Tổng chẵn ⇔ số số hạng lẻ là chẵn. Số chẵn không ảnh hưởng tính chẵn lẻ của tổng.

In [2]:
for group in ([2, 4], [1, 3], [1, 2], [1, 3, 5], [1, 3, 5, 7, 2]):
    odd = sum(x % 2 for x in group)
    print(group, "số lẻ =", odd, "-> tổng", sum(group), "chẵn" if sum(group) % 2 == 0 else "lẻ")

[2, 4] số lẻ = 0 -> tổng 6 chẵn
[1, 3] số lẻ = 2 -> tổng 4 chẵn
[1, 2] số lẻ = 1 -> tổng 3 lẻ
[1, 3, 5] số lẻ = 3 -> tổng 9 lẻ
[1, 3, 5, 7, 2] số lẻ = 4 -> tổng 18 chẵn


### 3.2 Dãy con ⇒ tập con

Dãy con giữ thứ tự nhưng tổng không phụ thuộc thứ tự ⇒ bài chỉ hỏi về **tập** `k` chỉ số. Ta được tự do **sắp xếp**.

### 3.3 Prefix sum trên mảng đã sắp giảm dần

`P[i]` = tổng `i` phần tử lớn nhất của một nhóm — truy vấn "tổng `i` số lớn nhất" trong `O(1)`.

In [3]:
nums = [4, 1, 5, 3, 1]
odds = sorted((x for x in nums if x % 2), reverse=True)
evens = sorted((x for x in nums if x % 2 == 0), reverse=True)
PO = [0, *accumulate(odds)]; PE = [0, *accumulate(evens)]
print("lẻ:", odds, "PO =", PO, "| chẵn:", evens, "PE =", PE)

lẻ: [5, 3, 1, 1] PO = [0, 5, 8, 9, 10] | chẵn: [4] PE = [0, 4]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `k = 3`. Lấy 3 số lớn nhất `[5, 4, 3]` ⇒ tổng `12` — đã chẵn ⇒ xong.

Thử `nums = [5, 4, 3, 1, 1]`, `k = 2`: 2 số lớn nhất `[5, 4]` tổng `9` (lẻ). Hai cách sửa:
- **(a)** bỏ số **lẻ** nhỏ nhất trong nhóm đã chọn (`5`), thêm số **chẵn** lớn nhất ngoài nhóm (không có) ⇒ không làm được;
- **(b)** bỏ số **chẵn** nhỏ nhất trong nhóm (`4`), thêm số **lẻ** lớn nhất ngoài nhóm (`3`) ⇒ `5 + 3 = 8`.

Đáp án `8`. Bảng theo số lượng số lẻ `i` (phải chẵn) — cách nhìn của Cách 4:

| `i` (số lẻ) | `k − i` (số chẵn) | tổng | hợp lệ? |
|---|---|---|---|
| 0 | 2 | cần 2 số chẵn, chỉ có 1 | ✗ |
| 2 | 0 | `5 + 3 = 8` | ✓ |

In [4]:
nums, k = [5, 4, 3, 1, 1], 2
odds = sorted((x for x in nums if x % 2), reverse=True); evens = sorted((x for x in nums if x % 2 == 0), reverse=True)
for i in range(0, k + 1, 2):
    ok = i <= len(odds) and k - i <= len(evens)
    print(f"i = {i}: {'tổng = ' + str(sum(odds[:i]) + sum(evens[:k - i])) if ok else 'không đủ phần tử'}")

i = 0: không đủ phần tử
i = 2: tổng = 8


## 5. Cách 1 — Brute Force / baseline: mọi tổ hợp

In [5]:
def even_sum_brute(nums, k):
    best = -1
    for comb in combinations(nums, k):
        s = sum(comb)
        if s % 2 == 0 and s > best:
            best = s
    return best

for (nums, k), want in examples:
    print(even_sum_brute(nums, k), "kỳ vọng", want)

12 kỳ vọng 12
12 kỳ vọng 12
-1 kỳ vọng -1


**Độ phức tạp:** `O(C(n, k) · k)` — mũ; chỉ dùng cho `n ≤ ~16`.

**Nút thắt:** chỉ có **tính chẵn lẻ** của tổng là ràng buộc — nhớ được bằng 1 bit, không cần liệt kê tổ hợp.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (DP):** duyệt từng phần tử, trạng thái `(j, p)` = đã chọn `j` phần tử, tổng có tính chẵn lẻ `p`; giá trị = tổng lớn nhất.

**Insight 2 (tham lam):** gọi `S` = tập `k` số lớn nhất. Nếu `sum(S)` chẵn ⇒ tối ưu (không tập nào lớn hơn). Nếu lẻ, đáp án là tập "gần `S` nhất" có tổng chẵn — chỉ cần **đổi một cặp**:
- bỏ số lẻ nhỏ nhất trong `S`, thêm số chẵn lớn nhất ngoài `S`; hoặc
- bỏ số chẵn nhỏ nhất trong `S`, thêm số lẻ lớn nhất ngoài `S`.

Lấy phương án tốt hơn (nếu tồn tại), không có thì `-1`.

**Insight 3 (tách nhóm):** tổng chẵn ⇔ chọn số lẻ với số lượng `i` chẵn. Với `i` cố định, tốt nhất là `i` số lẻ lớn nhất + `k − i` số chẵn lớn nhất. Thử mọi `i` chẵn hợp lệ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP `(số lượng, tính chẵn lẻ)`

In [6]:
def even_sum_dp(nums, k):
    NEG = float("-inf")
    best = [[NEG, NEG] for _ in range(k + 1)]     # best[j][p]
    best[0][0] = 0
    for x in nums:
        for j in range(min(k, 10**9), 0, -1):     # duyệt j giảm dần để mỗi phần tử dùng một lần
            for p in (0, 1):
                prev = best[j - 1][p ^ (x & 1)]
                if prev != NEG and prev + x > best[j][p]:
                    best[j][p] = prev + x
    return -1 if best[k][0] == NEG else best[k][0]

for (nums, k), want in examples:
    print(even_sum_dp(nums, k), "kỳ vọng", want)

12 kỳ vọng 12
12 kỳ vọng 12
-1 kỳ vọng -1


**Độ phức tạp:** `O(n · k)` thời gian (tệ nhất `10¹⁰` — chỉ để kiểm chứng), `O(k)` bộ nhớ.

### 7.2 Cách 3 — Tham lam + một lần đổi (code nộp)

In [7]:
def even_sum_greedy(nums, k):
    a = sorted(nums, reverse=True)
    top, rest = a[:k], a[k:]
    s = sum(top)
    if s % 2 == 0:
        return s
    INF = float("inf")
    min_odd_in = min((x for x in top if x % 2), default=None)
    min_even_in = min((x for x in top if x % 2 == 0), default=None)
    max_odd_out = max((x for x in rest if x % 2), default=None)
    max_even_out = max((x for x in rest if x % 2 == 0), default=None)
    best = -1
    if min_odd_in is not None and max_even_out is not None:
        best = max(best, s - min_odd_in + max_even_out)
    if min_even_in is not None and max_odd_out is not None:
        best = max(best, s - min_even_in + max_odd_out)
    return best

for (nums, k), want in examples:
    print(even_sum_greedy(nums, k), "kỳ vọng", want)

12 kỳ vọng 12
12 kỳ vọng 12
-1 kỳ vọng -1


Lưu ý: vì `sum(top)` lẻ, `top` chắc chắn có ít nhất một số lẻ (`min_odd_in` luôn tồn tại); nhưng có thể **không** có số chẵn trong `top`, hoặc không có số chẵn/lẻ phù hợp ngoài `top`.

Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** `O(n log n)` (sắp xếp), `O(n)` bộ nhớ.

### 7.3 Cách 4 — Tách chẵn/lẻ + prefix sum

In [8]:
def even_sum_split(nums, k):
    odds = sorted((x for x in nums if x & 1), reverse=True)
    evens = sorted((x for x in nums if not x & 1), reverse=True)
    PO = [0, *accumulate(odds)]; PE = [0, *accumulate(evens)]
    best = -1
    for i in range(0, min(k, len(odds)) + 1, 2):  # số lẻ được chọn: chẵn
        if k - i <= len(evens):
            best = max(best, PO[i] + PE[k - i])
    return best

for (nums, k), want in examples:
    print(even_sum_split(nums, k), "kỳ vọng", want)

12 kỳ vọng 12
12 kỳ vọng 12
-1 kỳ vọng -1


**Độ phức tạp:** `O(n log n)` (sắp hai nhóm) + `O(k)` để thử `i` ⇒ `O(n log n)`. Space `O(n)`.

Cách này không cần chứng minh "một lần đổi là đủ" — nó **duyệt hết** mọi phân bổ hợp lệ giữa lẻ và chẵn, mỗi phân bổ chọn tối ưu cục bộ (lớn nhất trong nhóm). Dễ mở rộng cho "tổng chia hết cho 3" (nhóm theo `x mod 3`)? Không trực tiếp — với 3 nhóm số dư cần duyệt hai chiều; xem Bài 5.

## 8. TRACE từng bước

Trace Cách 3 trên vài bộ: tập `k` số lớn nhất, tính chẵn lẻ, hai phương án đổi.

In [9]:
def trace_greedy(nums, k):
    a = sorted(nums, reverse=True); top, rest = a[:k], a[k:]; s = sum(top)
    print(f"nums = {nums}, k = {k}: top = {top}, rest = {rest}, tổng = {s} ({'chẵn' if s % 2 == 0 else 'lẻ'})")
    if s % 2 == 0:
        print("  -> trả về", s); return s
    moi = min((x for x in top if x % 2), default=None); mei = min((x for x in top if x % 2 == 0), default=None)
    moo = max((x for x in rest if x % 2), default=None); meo = max((x for x in rest if x % 2 == 0), default=None)
    a1 = s - moi + meo if moi is not None and meo is not None else None
    a2 = s - mei + moo if mei is not None and moo is not None else None
    print(f"  (a) bỏ lẻ nhỏ nhất trong {moi}, thêm chẵn lớn nhất ngoài {meo} -> {a1}")
    print(f"  (b) bỏ chẵn nhỏ nhất trong {mei}, thêm lẻ lớn nhất ngoài {moo} -> {a2}")
    r = max([v for v in (a1, a2) if v is not None], default=-1); print("  -> trả về", r); return r

for nums, k in [([4, 1, 5, 3, 1], 3), ([5, 4, 3, 1, 1], 2), ([9, 8, 2, 1], 2), ([7, 7, 7], 1), ([6, 3, 2, 2], 2)]:
    trace_greedy(nums, k)

nums = [4, 1, 5, 3, 1], k = 3: top = [5, 4, 3], rest = [1, 1], tổng = 12 (chẵn)
  -> trả về 12
nums = [5, 4, 3, 1, 1], k = 2: top = [5, 4], rest = [3, 1, 1], tổng = 9 (lẻ)
  (a) bỏ lẻ nhỏ nhất trong 5, thêm chẵn lớn nhất ngoài None -> None
  (b) bỏ chẵn nhỏ nhất trong 4, thêm lẻ lớn nhất ngoài 3 -> 8
  -> trả về 8
nums = [9, 8, 2, 1], k = 2: top = [9, 8], rest = [2, 1], tổng = 17 (lẻ)
  (a) bỏ lẻ nhỏ nhất trong 9, thêm chẵn lớn nhất ngoài 2 -> 10
  (b) bỏ chẵn nhỏ nhất trong 8, thêm lẻ lớn nhất ngoài 1 -> 10
  -> trả về 10
nums = [7, 7, 7], k = 1: top = [7], rest = [7, 7], tổng = 7 (lẻ)
  (a) bỏ lẻ nhỏ nhất trong 7, thêm chẵn lớn nhất ngoài None -> None
  (b) bỏ chẵn nhỏ nhất trong None, thêm lẻ lớn nhất ngoài 7 -> None
  -> trả về -1
nums = [6, 3, 2, 2], k = 2: top = [6, 3], rest = [2, 2], tổng = 9 (lẻ)
  (a) bỏ lẻ nhỏ nhất trong 3, thêm chẵn lớn nhất ngoài 2 -> 8
  (b) bỏ chẵn nhỏ nhất trong 6, thêm lẻ lớn nhất ngoài None -> None
  -> trả về 8


In [10]:
def trace_split(nums, k):
    odds = sorted((x for x in nums if x & 1), reverse=True); evens = sorted((x for x in nums if not x & 1), reverse=True)
    PO = [0, *accumulate(odds)]; PE = [0, *accumulate(evens)]
    print(f"lẻ = {odds}, chẵn = {evens}")
    for i in range(0, k + 1, 2):
        if i <= len(odds) and k - i <= len(evens):
            print(f"  i = {i}: PO[{i}] + PE[{k - i}] = {PO[i]} + {PE[k - i]} = {PO[i] + PE[k - i]}")
        else:
            print(f"  i = {i}: không đủ (cần {i} lẻ, {k - i} chẵn)")

trace_split([9, 8, 7, 6, 5, 2, 1], 4)

lẻ = [9, 7, 5, 1], chẵn = [8, 6, 2]
  i = 0: không đủ (cần 0 lẻ, 4 chẵn)
  i = 2: PO[2] + PE[2] = 16 + 14 = 30
  i = 4: PO[4] + PE[0] = 22 + 0 = 22


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def largestEvenSum(self, nums: List[int], k: int) -> int:
        a = sorted(nums, reverse=True)
        s = sum(a[:k])
        if s % 2 == 0:
            return s
        INF = float("inf")
        min_odd_in = min_even_in = INF
        for x in a[:k]:
            if x & 1:
                min_odd_in = min(min_odd_in, x)
            else:
                min_even_in = min(min_even_in, x)
        max_odd_out = max_even_out = -INF
        for x in a[k:]:
            if x & 1:
                max_odd_out = max(max_odd_out, x)
            else:
                max_even_out = max(max_even_out, x)
        ans = -1
        if min_odd_in < INF and max_even_out > -INF:
            ans = max(ans, s - min_odd_in + max_even_out)
        if min_even_in < INF and max_odd_out > -INF:
            ans = max(ans, s - min_even_in + max_odd_out)
        return ans

sol = Solution()
for (nums, k), want in examples:
    got = sol.largestEvenSum(nums, k)
    print(got, "OK" if got == want else "SAI")

12 OK
12 OK
-1 OK


## 10. Vì sao thuật toán đúng

**Cách 4 (trực tiếp nhất).** Mọi tập hợp lệ gồm `i` số lẻ (`i` chẵn) và `k − i` số chẵn. Với `i` cố định, thay bất kỳ số lẻ được chọn bằng số lẻ lớn hơn chưa chọn không làm giảm tổng và giữ tính chẵn lẻ ⇒ tối ưu là `i` số lẻ lớn nhất; tương tự cho số chẵn. Thử mọi `i` ⇒ đúng.

**Cách 3 (một lần đổi là đủ).** Gọi `S` = `k` số lớn nhất, `sum(S)` lẻ, và `T` là tập tối ưu (tổng chẵn). Gọi `i_S`, `i_T` là số số lẻ trong `S`, `T`; `i_S` lẻ, `i_T` chẵn ⇒ khác nhau.
- Nếu `i_T < i_S`: theo Cách 4, `T` là `i_T` số lẻ lớn nhất + `k − i_T` số chẵn lớn nhất. Hàm `f(i) = PO[i] + PE[k − i]` là **lõm** (hiệu `f(i+1) − f(i) = odd[i] − even[k−i−1]` giảm dần theo `i`) và đạt cực đại tại `i_S` (vì `S` là tập lớn nhất tuyệt đối). Trong các `i` chẵn, cực đại đạt ở `i_S − 1` hoặc `i_S + 1` — đúng hai phép đổi (a) (`i_S − 1`) và (b) (`i_S + 1`) với **một** cặp (tổng từ `f(i_S)` sang `f(i_S ± 1)`).
- Trường hợp `i_T > i_S` đối xứng.

⇒ Đáp án `= max(f(i_S − 1), f(i_S + 1))` (nếu tồn tại) — đúng giá trị Cách 3 tính. ✓

**DP.** Quy hoạch động 0/1 chuẩn: duyệt `j` giảm dần để mỗi phần tử dùng một lần; trạng thái chẵn lẻ chuyển `p ^ (x & 1)`. ✓

In [12]:
# Kiểm chứng tính lõm của f(i) = PO[i] + PE[k - i] trên dữ liệu ngẫu nhiên
rng = random.Random(10)
for _ in range(2000):
    nums = [rng.randint(0, 50) for _ in range(rng.randint(1, 12))]; k = rng.randint(1, len(nums))
    odds = sorted((x for x in nums if x & 1), reverse=True); evens = sorted((x for x in nums if not x & 1), reverse=True)
    PO = [0, *accumulate(odds)]; PE = [0, *accumulate(evens)]
    f = [PO[i] + PE[k - i] for i in range(k + 1) if i <= len(odds) and k - i <= len(evens)]
    diffs = [b - a for a, b in zip(f, f[1:])]
    assert all(d1 >= d2 for d1, d2 in zip(diffs, diffs[1:]))
print("f(i) lõm trên 2000 bộ ngẫu nhiên (hiệu liên tiếp không tăng)")

f(i) lõm trên 2000 bộ ngẫu nhiên (hiệu liên tiếp không tăng)


## 11. Phân tích Time / Space

`n ≤ 10⁵`, `k ≤ n`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi tổ hợp | `O(C(n,k) · k)` | `O(k)` |
| 2. DP chẵn lẻ | `O(n k)` | `O(k)` |
| 3. Tham lam + đổi | `O(n log n)` | `O(n)` |
| 4. Tách nhóm + prefix | `O(n log n)` | `O(n)` |

(Có thể đạt `O(n)` trung bình bằng `heapq.nlargest`/quickselect cho `k` phần tử lớn nhất, nhưng không cần thiết.)

In [13]:
rng = random.Random(1)
big = [rng.randint(0, 10**5) for _ in range(10**5)]
for k in (1, 500, 50000, 10**5):
    res = []
    for name, fn in [("greedy", even_sum_greedy), ("split", even_sum_split), ("submit", Solution().largestEvenSum)]:
        t0 = time.perf_counter(); r = fn(big, k); res.append(r)
        print(f"k = {k:>6} {name:7s}: {r:>11}  {(time.perf_counter() - t0) * 1000:.1f} ms")
    assert len(set(res)) == 1
t0 = time.perf_counter(); even_sum_dp(big[:2000], 200); print(f"dp n = 2000, k = 200: {time.perf_counter() - t0:.2f}s")

k =      1 greedy :       99998  29.9 ms
k =      1 split  :       99998  33.6 ms
k =      1 submit :       99998  21.3 ms


k =    500 greedy :    49887990  18.6 ms
k =    500 split  :    49887990  27.5 ms
k =    500 submit :    49887990  15.6 ms


k =  50000 greedy :  3747084608  18.8 ms


k =  50000 split  :  3747084608  29.2 ms
k =  50000 submit :  3747084608  17.3 ms
k = 100000 greedy :          -1  31.0 ms


k = 100000 split  :          -1  28.5 ms
k = 100000 submit :          -1  26.1 ms
dp n = 2000, k = 200: 0.06s


## 12. Bộ test

In [14]:
fns = [even_sum_dp, even_sum_greedy, even_sum_split, Solution().largestEvenSum]

for (nums, k), want in examples:
    assert even_sum_brute(nums, k) == want
    for fn in fns:
        assert fn(nums, k) == want

edge = [
    (([0], 1), 0), (([1], 1), -1), (([2], 1), 2),
    (([1, 1], 2), 2), (([1, 2], 2), -1), (([1, 2], 1), 2),
    (([5, 4, 3, 1, 1], 2), 8), (([9, 8, 2, 1], 2), 10),      # (a) và (b) đều có, (b): 9+1 = 10 vs (a) 8+2 = 10
    (([6, 3, 2, 2], 2), 8), (([7, 7, 7], 1), -1), (([7, 7, 7], 2), 14), (([7, 7, 7], 3), -1),
]
for (nums, k), want in edge:
    assert even_sum_brute(nums, k) == want, (nums, k)
    for fn in fns:
        assert fn(nums, k) == want, (fn, nums, k)

rng = random.Random(2024)
for _ in range(2000):
    n = rng.randint(1, 12)
    nums = [rng.randint(0, rng.choice([3, 10, 100])) for _ in range(n)]
    k = rng.randint(1, n)
    want = even_sum_brute(nums, k)
    for fn in fns:
        assert fn(nums, k) == want, (fn, nums, k)
print("ví dụ + edge + 2000 bộ ngẫu nhiên khớp brute force")

ví dụ + edge + 2000 bộ ngẫu nhiên khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ thử **một** kiểu hoán đổi

In [15]:
def wrong_one_swap(nums, k):
    a = sorted(nums, reverse=True); top, rest = a[:k], a[k:]; s = sum(top)
    if s % 2 == 0: return s
    moi = min((x for x in top if x % 2), default=None); meo = max((x for x in rest if x % 2 == 0), default=None)
    return s - moi + meo if moi is not None and meo is not None else -1     # SAI: thiếu kiểu (b)

for nums, k in [([5, 4, 3, 1, 1], 2), ([4, 1, 5, 3, 1], 3), ([9, 6, 5], 2)]:
    print(f"{nums}, k = {k}: sai = {wrong_one_swap(nums, k)} | đúng = {even_sum_brute(nums, k)}")

[5, 4, 3, 1, 1], k = 2: sai = -1 | đúng = 8
[4, 1, 5, 3, 1], k = 3: sai = 12 | đúng = 12
[9, 6, 5], k = 2: sai = -1 | đúng = 14


`[5, 4, 3, 1, 1]`, `k = 2`: không có số chẵn ngoài `top` ⇒ kiểu (a) bất khả, nhưng kiểu (b) cho `8`. `[9, 6, 5]`, `k = 2`: (a) bỏ `9` thêm — không có chẵn ngoài; (b) bỏ `6` thêm `5` ⇒ `14`. **Sửa:** xét cả hai và lấy max.

### Lỗi 2 — Đổi số **lớn nhất** thay vì nhỏ nhất trong `top` (hoặc nhỏ nhất ngoài `top`)

In [16]:
def wrong_pick_extremes(nums, k):
    a = sorted(nums, reverse=True); top, rest = a[:k], a[k:]; s = sum(top)
    if s % 2 == 0: return s
    best = -1
    oi = [x for x in top if x % 2]; ei = [x for x in top if x % 2 == 0]
    oo = [x for x in rest if x % 2]; eo = [x for x in rest if x % 2 == 0]
    if oi and eo: best = max(best, s - max(oi) + max(eo))          # SAI: bỏ số lẻ LỚN nhất
    if ei and oo: best = max(best, s - max(ei) + max(oo))          # SAI: bỏ số chẵn LỚN nhất
    return best

rng = random.Random(3); shown = 0
for _ in range(5000):
    nums = [rng.randint(0, 20) for _ in range(rng.randint(2, 8))]; k = rng.randint(1, len(nums))
    if wrong_pick_extremes(nums, k) != even_sum_brute(nums, k):
        print(f"phản ví dụ: {nums}, k = {k} | sai = {wrong_pick_extremes(nums, k)} | đúng = {even_sum_brute(nums, k)}")
        shown += 1
        if shown == 2: break

phản ví dụ: [0, 2, 5, 18, 1], k = 3 | sai = 20 | đúng = 24
phản ví dụ: [16, 12, 20, 19, 17, 3, 19, 16], k = 5 | sai = 88 | đúng = 90


Muốn mất ít nhất ⇒ bỏ phần tử **nhỏ nhất** (đúng tính chẵn lẻ) trong `top`, thêm phần tử **lớn nhất** (đúng tính chẵn lẻ) ngoài `top`. **Sửa:** `min` trong, `max` ngoài.

### Lỗi 3 — Thử cả số lẻ `i` lẻ (quên điều kiện chẵn) trong Cách 4

In [17]:
def wrong_all_i(nums, k):
    odds = sorted((x for x in nums if x & 1), reverse=True); evens = sorted((x for x in nums if not x & 1), reverse=True)
    PO = [0, *accumulate(odds)]; PE = [0, *accumulate(evens)]; best = -1
    for i in range(0, min(k, len(odds)) + 1):                     # SAI: bước 1 — có cả i lẻ
        if k - i <= len(evens): best = max(best, PO[i] + PE[k - i])
    return best

for (nums, k), want in examples + [(([5, 4, 3, 1, 1], 2), 8)]:
    print(f"{nums}, k = {k}: sai = {wrong_all_i(nums, k)} (chẵn? {wrong_all_i(nums, k) % 2 == 0}) | đúng = {want}")

[4, 1, 5, 3, 1], k = 3: sai = 12 (chẵn? True) | đúng = 12
[4, 6, 2], k = 3: sai = 12 (chẵn? True) | đúng = 12
[1, 3, 5], k = 1: sai = 5 (chẵn? False) | đúng = -1
[5, 4, 3, 1, 1], k = 2: sai = 9 (chẵn? False) | đúng = 8


Với `i` lẻ tổng là **lẻ** ⇒ không hợp lệ. Ví dụ 3 trả `5` (lẻ!) thay vì `-1`. **Sửa:** `range(0, …, 2)`.

### Lỗi 4 — Bỏ phần tử nhỏ nhất của `top` và thêm phần tử kế tiếp (không quan tâm tính chẵn lẻ)

In [18]:
def wrong_next_element(nums, k):
    a = sorted(nums, reverse=True); s = sum(a[:k])
    if s % 2 == 0: return s
    for j in range(k, len(a)):                                      # SAI: thêm a[j], bỏ a[k-1]
        t = s - a[k - 1] + a[j]
        if t % 2 == 0: return t
    return -1

rng = random.Random(8); shown = 0
for _ in range(20000):
    nums = [rng.randint(0, 20) for _ in range(rng.randint(2, 7))]; k = rng.randint(1, len(nums))
    if wrong_next_element(nums, k) != even_sum_brute(nums, k):
        print(f"phản ví dụ: {nums}, k = {k} | sai = {wrong_next_element(nums, k)} | đúng = {even_sum_brute(nums, k)}")
        shown += 1
        if shown == 2: break

phản ví dụ: [15, 18, 6, 12, 2], k = 4 | sai = -1 | đúng = 38
phản ví dụ: [6, 9, 4, 11, 16, 9], k = 5 | sai = -1 | đúng = 46


Phần tử bị bỏ phải **khác tính chẵn lẻ** với phần tử được thêm; bỏ cố định `a[k − 1]` có thể không đổi được tính chẵn lẻ, hoặc bỏ nhầm phần tử lớn. **Sửa:** dùng đúng hai phương án (a), (b).

### Lỗi 5 — Nghĩ rằng `k ≤ n` thì luôn có đáp án

`[1, 2]`, `k = 2` ⇒ tổng duy nhất `3` lẻ ⇒ `-1`. `[7, 7, 7]`, `k = 3` ⇒ `-1`. Luôn trả `-1` khi cả hai phương án đều bất khả (code nộp khởi tạo `ans = -1`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Dãy con độ dài `k`, tối ưu tổng" ⇒ thứ tự không quan trọng, **sắp xếp** được.
- Ràng buộc chẵn lẻ (hoặc modulo nhỏ) ⇒ (a) DP `(số lượng, số dư)`, hoặc (b) "lấy tốt nhất rồi sửa bằng ít thay đổi nhất".
- Khi "sửa", luôn liệt kê **mọi** kiểu sửa tối thiểu và kiểm tra tồn tại.
- Tách theo lớp (chẵn/lẻ) + prefix sum ⇒ duyệt phân bổ số lượng giữa các lớp.
- Họ bài: 1262 *Greatest Sum Divisible by Three*, 1984 *Minimum Difference Between Highest and Lowest of K Scores*, 2099 *Find Subsequence of Length K With the Largest Sum*, 2551 *Put Marbles in Bags*.

## 15. Biến thể của bài

1. **Tổng lẻ lớn nhất** độ dài `k`: Cách 4 với `i` lẻ; Cách 3 khi `top` chẵn thì đổi.
2. **Không cố định độ dài** — tổng chẵn lớn nhất của dãy con bất kỳ: lấy mọi số; nếu lẻ, bỏ số lẻ nhỏ nhất.
3. **Tổng chia hết cho `m`** với đúng `k` phần tử: DP `(j, r)` `O(n k m)` (Bài 5).
4. **2099:** dãy con độ dài `k` tổng lớn nhất **trả về chính dãy con** (giữ thứ tự gốc).

In [19]:
def largest_odd_sum(nums, k):
    """Biến thể 1."""
    odds = sorted((x for x in nums if x & 1), reverse=True); evens = sorted((x for x in nums if not x & 1), reverse=True)
    PO = [0, *accumulate(odds)]; PE = [0, *accumulate(evens)]; best = -1
    for i in range(1, min(k, len(odds)) + 1, 2):
        if k - i <= len(evens): best = max(best, PO[i] + PE[k - i])
    return best

def largest_even_any_length(nums):
    """Biến thể 2 (dãy con không rỗng)."""
    s = sum(nums)
    if s % 2 == 0: return s
    return s - min(x for x in nums if x & 1) if len(nums) > 1 else -1

def subsequence_k_largest(nums, k):
    """Biến thể 4 (2099): giữ thứ tự gốc."""
    idx = sorted(sorted(range(len(nums)), key=lambda i: -nums[i])[:k])
    return [nums[i] for i in idx]

rng = random.Random(15)
for _ in range(500):
    nums = [rng.randint(0, 20) for _ in range(rng.randint(1, 10))]; k = rng.randint(1, len(nums))
    bo = max((sum(c) for c in combinations(nums, k) if sum(c) % 2), default=-1)
    assert largest_odd_sum(nums, k) == bo
print("tổng lẻ lớn nhất ví dụ 1:", largest_odd_sum([4, 1, 5, 3, 1], 3), "| 2099:", subsequence_k_largest([2, 1, 3, 3], 2))

tổng lẻ lớn nhất ví dụ 1: 9 | 2099: [3, 3]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tổ hợp | định nghĩa | mũ | `O(k)` | kiểm chứng |
| 2. DP chẵn lẻ | `(j, p)` | `O(n k)` | `O(k)` | ràng buộc modulo tổng quát |
| 3. Tham lam + đổi | sửa tối thiểu | `O(n log n)` | `O(n)` | **nộp bài** |
| 4. Tách nhóm + prefix | duyệt số lẻ `i` | `O(n log n)` | `O(n)` | dễ chứng minh, ít lỗi |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `[2, 2, 2]`, `k = 2`? `[3, 3, 2, 1]`, `k = 3`? `[1, 1, 1, 1, 6]`, `k = 2`?

**Bài 2 (tính tay):** `nums = [10, 9, 7, 4, 3]`, `k = 3`. Viết `top`, tổng, hai phương án đổi và đáp án.

**Bài 3 (điền TODO):** hoàn thiện `parity_split(nums)` trả về `(odds_desc, evens_desc)` — hai danh sách đã sắp giảm dần.

**Bài 4 (tự viết lại):** viết `largest_even_subsequence(nums, k)` trả về **chính dãy con** (giữ thứ tự gốc) đạt tổng chẵn lớn nhất, hoặc `None`.

**Bài 5 (mini challenge):** tổng lớn nhất **chia hết cho 3** của dãy con độ dài đúng `k`, hoặc `-1`. Viết DP `O(n · k · 3)` và kiểm chứng bằng brute force. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [20]:
def parity_split(nums):
    # TODO: lọc lẻ/chẵn rồi sorted(..., reverse=True)
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `[2,2,2]` ⇒ `4`. (2) `[3,3,2,1]`, `k = 3`: `top = [3,3,2]` tổng `8` chẵn. (3) `[1,1,1,1,6]`, `k = 2`: `top = [6,1]` lẻ; (a) bỏ `1` thêm chẵn — không có; (b) bỏ `6` thêm `1` ⇒ `2`.

**Bài 2:** (1) `top = [10, 9, 7]`, tổng `26` chẵn. (2) Không cần đổi. (3) Đáp án `26`.

**Bài 3:** (1) Hai list comprehension. (2) `x & 1` để kiểm tra lẻ. (3) Sắp giảm dần.

**Bài 4:** (1) Dùng Cách 4 để biết `i` tối ưu. (2) Lấy chỉ số của `i` số lẻ lớn nhất và `k − i` số chẵn lớn nhất. (3) Sắp lại chỉ số tăng dần, trả về các giá trị.

**Bài 5:** (1) `best[j][r]`, `r = tổng mod 3`. (2) Chuyển `best[j][r] ← best[j−1][(r − x) mod 3] + x`. (3) Duyệt `j` giảm dần.

## 19. Đáp án tham khảo

In [21]:
# Bài 1
assert [Solution().largestEvenSum(*a) for a in [([2, 2, 2], 2), ([3, 3, 2, 1], 3), ([1, 1, 1, 1, 6], 2)]] == [4, 8, 2]
# Bài 2
assert trace_greedy([10, 9, 7, 4, 3], 3) == 26

# Bài 3
def parity_split_ans(nums):
    return sorted((x for x in nums if x & 1), reverse=True), sorted((x for x in nums if not x & 1), reverse=True)

# Bài 4
def largest_even_subsequence_ans(nums, k):
    oi = sorted((i for i in range(len(nums)) if nums[i] & 1), key=lambda i: -nums[i])
    ei = sorted((i for i in range(len(nums)) if not nums[i] & 1), key=lambda i: -nums[i])
    best, best_i = -1, None
    for i in range(0, min(k, len(oi)) + 1, 2):
        if k - i <= len(ei):
            s = sum(nums[j] for j in oi[:i]) + sum(nums[j] for j in ei[:k - i])
            if s > best: best, best_i = s, i
    if best_i is None: return None
    idx = sorted(oi[:best_i] + ei[:k - best_i])
    return [nums[j] for j in idx]

rng = random.Random(41)
for _ in range(1000):
    nums = [rng.randint(0, 30) for _ in range(rng.randint(1, 10))]; k = rng.randint(1, len(nums))
    o, e = parity_split_ans(nums)
    assert sorted(o + e, reverse=True) == sorted(nums, reverse=True)
    sub = largest_even_subsequence_ans(nums, k); want = even_sum_brute(nums, k)
    if want == -1:
        assert sub is None
    else:
        assert len(sub) == k and sum(sub) == want
        it = iter(nums); assert all(x in it for x in sub)        # đúng là dãy con (giữ thứ tự)
print("Bài 4 ví dụ 1:", largest_even_subsequence_ans([4, 1, 5, 3, 1], 3))
print("Bài 1-4: OK")

nums = [10, 9, 7, 4, 3], k = 3: top = [10, 9, 7], rest = [4, 3], tổng = 26 (chẵn)
  -> trả về 26
Bài 4 ví dụ 1: [4, 5, 3]
Bài 1-4: OK


In [22]:
# Bài 5 — tổng lớn nhất chia hết cho 3, đúng k phần tử
def largest_div3(nums, k):
    NEG = float("-inf"); best = [[NEG] * 3 for _ in range(k + 1)]; best[0][0] = 0
    for x in nums:
        for j in range(k, 0, -1):
            for r in range(3):
                prev = best[j - 1][(r - x) % 3]
                if prev != NEG and prev + x > best[j][r]:
                    best[j][r] = prev + x
    return -1 if best[k][0] == NEG else best[k][0]

def largest_div3_brute(nums, k):
    return max((sum(c) for c in combinations(nums, k) if sum(c) % 3 == 0), default=-1)

rng = random.Random(55)
for _ in range(800):
    nums = [rng.randint(0, 30) for _ in range(rng.randint(1, 10))]; k = rng.randint(1, len(nums))
    assert largest_div3(nums, k) == largest_div3_brute(nums, k)
print("Bài 5 ví dụ: [4, 1, 5, 3, 1], k = 3 ->", largest_div3([4, 1, 5, 3, 1], 3), "| k = 2 ->", largest_div3([4, 1, 5, 3, 1], 2))

Bài 5 ví dụ: [4, 1, 5, 3, 1], k = 3 -> 12 | k = 2 -> 9


## 20. Template nhớ nhanh + Checklist

```python
# Tổng chẵn lớn nhất của k phần tử: tách lẻ/chẵn, thử số lẻ i = 0, 2, 4, ...
odds  = sorted((x for x in nums if x & 1), reverse=True)
evens = sorted((x for x in nums if not x & 1), reverse=True)
PO, PE = [0, *accumulate(odds)], [0, *accumulate(evens)]
best = -1
for i in range(0, min(k, len(odds)) + 1, 2):
    if k - i <= len(evens):
        best = max(best, PO[i] + PE[k - i])
# Hoặc: top-k; nếu lẻ -> max(bỏ lẻ nhỏ nhất + thêm chẵn lớn nhất ngoài, bỏ chẵn nhỏ nhất + thêm lẻ lớn nhất ngoài)
```

**Checklist trước khi submit:**
- [ ] Tổng `top-k` chẵn ⇒ trả ngay.
- [ ] Xét **cả hai** kiểu đổi, kiểm tra tồn tại từng phần tử.
- [ ] Bỏ phần tử **nhỏ nhất** trong, thêm phần tử **lớn nhất** ngoài.
- [ ] Trả `-1` khi không phương án nào hợp lệ.
- [ ] Cách tách nhóm: chỉ thử `i` chẵn.